# 3.2 Het ontdekken van een cyclus die je vooraf niet wist

In les 3.1 trokken we cycli af waarvan je al wist dat ze bestonden: een dag heeft 24 uur en een week heeft zeven dagen, dus je wist vooraf precies welke periode je aan de ontbinding moest meegeven. Dit notebook is bedoeld voor situaties waarin je dat NIET weet — wanneer de data een ritme vertoont, maar niemand je heeft verteld hoe lang die cyclus duurt.

Drie technieken, in de volgorde waarin je ze in de praktijk zou inzetten:

1. **Autocorrelatie (ACF)** — correleer de tijdreeks met een verschoven kopie van zichzelf. De verschuiving (*lag*) die het sterkst correleert, is de lengte van de cyclus.
2. **Seizoensontbinding (Decomposition)** met die gemeten periode — dezelfde stap als in les 3.1, maar nu met een empirisch gemeten periode in plaats van een aanname.
3. **Fourier-transformaties**, voor wanneer het patroon niet uit één enkele cyclus bestaat, maar uit meerdere gelijktijdige golven.

Die laatste techniek komt aan bod omdat de eerste twee falen op de tweede dataset (zonnevlekken). En de manier waaróp ze falen is de meest waardevolle les van dit notebook.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from goad_toolkit.config import FileConfig
from goad_toolkit.filehandler import FileHandler
from goad_toolkit.visualizer import ACFPlot, DecomposePlot, PlotSettings
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf

Laten we wat synthetische data genereren. We creëren:

- 3 jaar aan dagelijkse data
- met 4 seizoenscycli per jaar (ongeveer elke 90 dagen)
- we voegen een opwaartse trend toe van +10 over de gehele periode
- en we voegen toevallige ruis toe


In [ ]:
# Generate a date range
years = 3
dates = pd.date_range(start="2020-01-01", periods=365 * years, freq="D")

# Generate synthetic data with seasonality
cycles = 4
np.random.seed(42)
trend = np.linspace(0, 10, len(dates))  # Linear trend
seasonality = 10 * np.sin(
    np.linspace(0, cycles * years * 2 * np.pi, len(dates))
)  # Yearly seasonality
noise = np.random.normal(0, 1, len(dates))  # Random noise
synthetic_data = trend + seasonality + noise

synthetic = pd.DataFrame({"Date": dates, "Value": synthetic_data}).set_index("Date")
plt.plot(synthetic.index, synthetic["Value"])
plt.xticks(rotation=45)
plt.show()

Autocorrelatie, vaak gevisualiseerd via Autocorrelatieplots (tsaplots), is een statistische maatstaf om de mate van samenhang te bepalen tussen een tijdreeks en een vertraagde versie van zichzelf (*lagged series*) over opeenvolgende tijdsintervallen. Het meet in feite hoe sterk huidige waarden gerelateerd zijn aan waarden uit het verleden. Deze relatie wordt uitgedrukt als een correlatiecoëfficiënt tussen -1 en 1, waarbij:

- **1** duidt op een perfecte positieve correlatie: stijgt de tijdreeks, dan stijgt de vertraagde reeks ook.
- **-1** duidt op een perfecte negatieve correlatie: stijgt de tijdreeks, dan daalt de vertraagde reeks, en vice versa.
- **0** duidt op geen correlatie: er is geen lineair verband tussen de tijdreeks en zijn vertraagde versie.

Waarom gebruik je autocorrelatie?

1. **Seizoenspatronen identificeren**: Als een tijdreeks seizoensinvloeden kent, toont de plot regelmatige, significante pieken bij lags die overeenkomen met de cycluslengte (bijv. pieken om de 7 dagen bij een weekritme, of om de 12 maanden bij jaardata).
2. **Modelkeuze**: Helpt bepalen welke tijdreeksmodellen geschikt zijn (zoals ARIMA) door de juiste orde van autoregressieve (AR) of moving average (MA) componenten te identificeren.
3. **Voorbewerking van data**: Bepalen of transformaties of differentiëren (*differencing*) nodig zijn om de tijdreeks stationair te maken.
4. **Opsporen van anomalieën**: Significante afwijkingen van het verwachte patroon kunnen wijzen op uitschieters, structurele breuken of fundamentele veranderingen in het proces.


In [ ]:
settings = PlotSettings(
    figsize=(10, 4),
    title="Autocorrelation of the synthetic series",
    xlabel="lag (days)",
    ylabel="correlation with the shifted copy",
)
fig, ax = ACFPlot(settings).plot(data=synthetic["Value"], nlags=365)

De cyclus die we hebben ingebouwd is direct zichtbaar: een piek rond lag 90 dagen (vier cycli per jaar van 365 dagen). De grijze zone is het betrouwbaarheidsinterval waarbinnen een correlatie statistisch niet te onderscheiden is van nul; lags die buiten deze blauwe/grijze band steken zijn de enige die er echt toe doen.

Lag 0 is altijd 1.0 — een reeks correleert immers perfect met zichzelf — en de eerste paar lags zijn hoog om een banale reden: de waarde van gisteren ligt meestal dicht bij die van vandaag. Om de *echte cyclus* te vinden, sla je dus de eerste daling over en zoek je de hoogste significante piek daarna.


In [ ]:
acf_values = acf(synthetic["Value"], nlags=365, fft=True)
skip = 50
lag = int(np.argmax(acf_values[skip:])) + skip
print(f"strongest lag after the first {skip}: {lag} days (r = {acf_values[lag]:.2f})")
print(f"we built the series with {cycles} cycles per year, so we expected "
      f"{round(365 / cycles)} days")

In [ ]:
settings = PlotSettings(
    figsize=(10, 10),
    title="Decomposition of the synthetic series",
    xlabel="",
    ylabel="value",
    max_cols=1,
)
fig = DecomposePlot(settings).plot(data=synthetic, column="Value", period=lag)
# Seaborn fills an empty xlabel with the index name, so every panel repeats "date" —
# straight into the title of the panel below it. Keep it on the bottom one only.
for panel in fig.axes[:-1]:
    panel.set_xlabel("")

Dit is een deconstructie van hoe we de tijdreeks hebben opgebouwd: een rechte trendlijn, een herhalende seizoensgolf en toevallige ruis. Let op: de periode (periode 91) kwam voort uit onze empirische meting hierboven, niet omdat we het antwoord vooraf wisten!

Als we de trend en de seizoensgolf aftrekken, houden we het residu over: datgene wat het model niet kon verklaren.


In [ ]:
result = seasonal_decompose(synthetic["Value"], model="additive", period=lag)
residual = synthetic["Value"] - result.seasonal - result.trend
plt.plot(residual)
plt.title("what is left after removing trend and season")
plt.show()
print(f"the residual keeps {residual.var() / synthetic['Value'].var():.1%} of the "
      f"variance; we put in noise with a variance of {noise.var():.2f}")

## 3.3 Zonnevlekken: waar deze klassieke methode spaak loopt

Maandelijkse zonnevlek-tellingen van 1749 tot heden, gepubliceerd door de Koninklijke Sterrenwacht van België. De zonnecyclus duurt ruwweg elf jaar — lang genoeg dat niemand het met het blote oog zou raden, en beroemd genoeg dat we het juiste antwoord extern kunnen controleren.

Het downloaden verloopt via Goad's `FileHandler` in plaats van een losse `requests.get`. Dat heeft één groot voordeel: het bestand wordt opgeslagen in `~/.cache/mads-dav/raw` en niet bij elke run opnieuw gedownload. Een notebook dat bij elke celuitvoering data van internet trekt, faalt in een lokaal met slechte wifi en is onbeleefd tegenover de server.


In [ ]:
config = FileConfig(
    data_dir=Path.home() / ".cache/mads-dav",
    filename=Path("sunspots.txt"),
    url="https://www.sidc.be/SILSO/DATA/SN_m_tot_V2.0.txt",
    date_column=None,
)
handler = FileHandler(config)
handler.download()

# `FileHandler.load` reads .csv and .parquet; this one is whitespace-separated with no
# header, so we read it ourselves and only use the handler for the fetching and caching.
sunspots = pd.read_csv(
    config.data_dir / "raw" / config.filename,
    sep=r"\s+",
    header=None,
    names=["year", "month", "decimal_date", "sunspots", "std_dev", "observations",
           "definitive"],
)
sunspots["date"] = pd.to_datetime({"year": sunspots.year, "month": sunspots.month, "day": 1})  # ty: ignore[no-matching-overload]
sunspots = sunspots.set_index("date")
print(f"{len(sunspots):,} monthly observations, {sunspots.index.min():%Y} to "
      f"{sunspots.index.max():%Y}")

plt.figure(figsize=(12, 5))
plt.plot(sunspots.index, sunspots["sunspots"])
plt.title("Monthly sunspot number")
plt.show()

In [ ]:
settings = PlotSettings(
    figsize=(10, 4),
    title="Autocorrelation of the sunspot number",
    xlabel="lag (months)",
    ylabel="correlation with the shifted copy",
)
fig, ax = ACFPlot(settings).plot(data=sunspots["sunspots"], nlags=30 * 12)

In [ ]:
acf_values = acf(sunspots["sunspots"], nlags=365, fft=True)
skip = 50
solar_lag = int(np.argmax(acf_values[skip:])) + skip
print(f"strongest lag after the first {skip}: {solar_lag} months "
      f"= {solar_lag / 12:.1f} years (r = {acf_values[solar_lag]:.2f})")

10,7 jaar — gevonden door een algoritme dat helemaal niets weet van de zon of sterrenkunde! De officiële wetenschappelijke literatuur stelt dat de cyclus ongeveer 11 jaar is, dus de meting is griezelig nauwkeurig. Dit is het enige moment in dit notebook waarop je een berekend getal kunt controleren aan de hand van de echte buitenwereld. Vier die overwinning, en kijk vervolgens wat er gebeurt als we deze periode aan de klassieke seizoensontbinding (`seasonal_decompose`) voeden...


In [ ]:
settings = PlotSettings(
    figsize=(10, 10),
    title=f"Sunspots decomposed with a {solar_lag}-month period",
    xlabel="",
    ylabel="sunspots",
    max_cols=1,
)
fig = DecomposePlot(settings).plot(data=sunspots, column="sunspots", period=solar_lag)
# Seaborn fills an empty xlabel with the index name, so every panel repeats "date" —
# straight into the title of the panel below it. Keep it on the bottom one only.
for panel in fig.axes[:-1]:
    panel.set_xlabel("")

In [ ]:
rows = []
for period in (solar_lag, 365):
    result = seasonal_decompose(sunspots["sunspots"], model="additive", period=period)
    residual = result.resid.dropna()
    residual_acf = acf(residual, nlags=200, fft=True)
    residual_lag = int(np.argmax(residual_acf[skip:])) + skip
    rows.append({
        "period (months)": period,
        "= years": round(period / 12, 1),
        "residual variance": residual.var() / sunspots["sunspots"].var(),
        "residual's own strongest lag": residual_lag,
        "r at that lag": round(residual_acf[residual_lag], 2),
    })
print(pd.DataFrame(rows).to_string(index=False,
                                   formatters={"residual variance": "{:.1%}".format}))

Kijk goed naar de tweede kolom van de correlatietabel. **Het residu behoudt ongeveer 70% van de oorspronkelijke variantie, en bevat zélf nog steeds een 11-jarige cyclus!** — datgene wat we zojuist hebben afgetrokken zit nog steeds volop in het residu, met een correlatie van 0,5 (nauwelijks zwakker dan in de ruwe data).

Dat ligt niet aan een verkeerd gekozen periode. Zelfs als je 365 maanden (30 jaar) kiest, verandert het nauwelijks. `seasonal_decompose` faalt hier om een fundamentele reden die geen enkele parameter kan oplossen: het middelt elke 128e observatie en noemt dat "het seizoen". Daarmee veronderstelt het model dat elke cyclus *exact even lang en exact dezelfde vorm* heeft als de vorige! Zonnecycli duren in werkelijkheid tussen de 9 en 14 jaar en stijgen veel steiler dan ze dalen. Als je die allemaal op één hoop veegt en middelt, krijg je een uitgesmeerde brij die op geen enkele echte cyclus past.

Het eerlijke wetenschappelijke rapport luidt dus: de cyclus is reëel en meetbaar, maar klassieke decompositie is het verkeerde gereedschap hiervoor. Dát is waarom de volgende paragraaf bestaat: een methode die niet uitgaat van één vaste golf van één vaste lengte.

> **De gouden regel van Les 3:** *Een methode die zonder foutmeldingen draait, heeft je nog geen gelijk gegeven!* `seasonal_decompose` produceerde vier keurige deelgrafiekjes voor beide periodes. Pas toen we het residu inspecteerden, zagen we dat het model er faliekant naast zat.


## 3.4 Fourier-transformaties (FT)


In [ ]:
import numpy as np
import pandas as pd
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt

Omdat de patronen in deze tijdreeks complexer zijn, kiezen we een andere benadering: **Fourier-transformaties**. Het kernidee achter Fourier is simpel: *elk willekeurig complex signaal kan worden ontleed in een optelsom van zuivere sinusgolven*.

De Fourier-transformatie neemt een grillig patroon en splitst het uit in een reeks sinussen met verschillende frequenties en amplitudes.

We illustreren dit eerst op onze synthetische data:


In [ ]:
y = synthetic_data.copy()
y -= y.mean()
plt.plot(y)

In [ ]:
def fourier_model(timeseries, k):
    # Calculate the number of data points in the timeseries
    t = 1.0
    n = len(timeseries)
    # Generate a time vector 'x' from 0 to n*t (excluding the endpoint) evenly spaced
    x = np.linspace(0.0, n * t, n, endpoint=False)
    # Perform the Fourier Transform of the timeseries
    yf = fft(timeseries)
    # Generate the frequency bins for the first half of the Fourier Transform result
    # This represents the positive frequencies up to the Nyquist frequency
    # the nyquist frequency is the highest frequency that can be represented in the fourier transform
    # it is half of the sampling frequency
    xf = fftfreq(n, t)[: n // 2]
    # Identify indices of the 'k' largest frequencies by their magnitude in the first half of the Fourier spectrum
    # the largest frequencies are the most important components of the signal
    indices = np.argsort(np.abs(yf[0 : n // 2]))[-k:]
    # Extract the frequencies corresponding to the 'k' largest magnitudes
    frequencies = xf[indices]
    # Calculate the amplitudes of these frequencies as twice the magnitude divided by n
    # This accounts for the symmetry of the Fourier Transform for real signals
    amplitudes = 2.0 / n * np.abs(yf[indices])
    # Extract the phases of these frequencies and adjust by adding pi/2 to align phases
    phases = np.angle(yf[indices]) + 1 / 2 * np.pi
    # Return a dictionary of the model parameters: 'x', 'frequencies', 'amplitudes', 'phases'
    return {
        "x": x,
        "frequencies": frequencies,
        "amplitudes": amplitudes,
        "phases": phases,
    }

Deze decompositie levert ons de 'top k' sinusgolven op met de hoogste amplitudes (en dus de meeste verklaringskracht), waarmee we de originele data kunnen reconstrueren (modelleren). Laten we ons Fourier-model visualiseren:


In [ ]:
def model(parameters):
    # Extract the time vector 'x' from the parameters
    x = parameters["x"]
    # Extract the frequencies, amplitudes, and phases from the parameters
    frequencies = parameters["frequencies"]
    amplitudes = parameters["amplitudes"]
    phases = parameters["phases"]

    # Initialize a zero array 'y' of the same shape as 'x' to store the model output
    y = np.zeros_like(x)

    # Add each sine wave component to 'y' based on the extracted frequencies, amplitudes, and phases
    for freq, amp, phase in zip(frequencies, amplitudes, phases):
        y += amp * np.sin(2.0 * np.pi * freq * x + phase)

    # Return the composite signal 'y' as the sum of the sine wave components
    return y

En vergelijk het met de originele data:


In [ ]:
k = 2  # Number of Fourier components

parameters = fourier_model(y, k)
x = parameters["x"]
y_model = model(parameters)


def plot_model(x, y, y_model):
    plt.figure(figsize=(15, 6))
    plt.plot(x, y, label="Original Data")
    plt.plot(x, y_model, label="Modeled with Fourier", linestyle="--")
    plt.legend()
    plt.show()


plot_model(x, y, y_model)

Het is niet 100% perfect, maar buitengewoon accuraat! Een prettige bijvangst is dat we het grootste deel van de toevallige ruis direct kwijt zijn.

Laten we nu exact dezelfde techniek loslaten op een deel van de zonnevlekken-dataset:


In [ ]:
y = sunspots["sunspots"].values.copy()
y -= y.mean()
y = y[: 12 * 50]  # first 50 years
plt.plot(y)  # first 50 years

In [ ]:
k = 9  # Number of Fourier components

parameters = fourier_model(y, k)
y_model = model(parameters)
x = parameters["x"]
plot_model(x, y, y_model)

Hoewel dit er visueel indrukwekkend uitziet, hebben we hier vrijwel zeker te maken met **overfitting** van de tijdreeks: het is zeer onwaarschijnlijk dat dit model betrouwbare voorspellingen doet voor de toekomst, omdat het model te gretig de toevallige ruis in de data heeft gefit. Desondanks is het een prachtige illustratie van hoe deze techniek werkt.

Je zou dit model kunnen verbeteren door:
- Het aantal sinusgolven (*k*) te verminderen om het model eenvoudiger en robuuster te maken.
- Het model te fitten op een trainingset (bijv. de eerste 50 jaar) en de voorspellende waarde te testen op een testset (de volgende 50 jaar).
- Ruis vooraf te verminderen met afvlakkingstechnieken (*smoothing*).
- Domeinkennis in te brengen (overleggen met zonnefysici).
- Rekening te houden met de langzame drift in de *piekhoogte* over eeuwen heen (die de decompositie in de trend stopte).
- Betere wiskundige manieren te vinden om de asymmetrische vorm van de zonnevlekken te modelleren dan simpelweg het gemiddelde aftrekken.

Hieronder vind je een korte crash-course in afvlakking (*smoothing*). Dit is echter een vak over data-visualisatie, dus we raken dit onderwerp slechts op hoofdlijnen aan. In het vak Machine Learning gaan we hier veel dieper op in en laten we neurale netwerken zelf uitzoeken wat de beste filters zijn.


## 3.6 Reflectievragen

1. **Welke periode heb je gebruikt, en waar kwam die vandaan?** — Gemeten via de autocorrelatieplot, of aangenomen omdat een week zeven dagen heeft? Beide zijn valide; ze met elkaar verwarren is dat niet.
2. **Het residu gecontroleerd?** Is het patroon dat je dacht af te trekken ook daadwerkelijk verdwenen? Bij de zonnevlekken bleek het residu nog volop seizoensinvloed te bevatten!

---

**Hoe we nu verder gaan:** Niet met een autocorrelatieplot naar je eigen WhatsApp-chat rennen! Een persoonlijke chat verbergt zelden een mysterieuze cyclus die niemand wist, en ernaar zoeken met deze zware wiskunde is *"een probleem zoeken bij een oplossing"*. In [03.3-events-in-your-chat](03.3-events-in-your-chat.ipynb) draaien we de vraag om: begin bij een gebeurtenis waarvan je ZEKER WEET dat hij plaatsvond, formuleer vooraf welke impact dat zou moeten hebben, en bouw dán pas de visualisatie om dat residu aan te tonen.
